<a href="https://colab.research.google.com/github/andrealii/retail-site-failure-early-warning/blob/main/01_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Data Pipeline & Session Feature Engineering
**Project:** Retail Site Failure Early-Warning System  
**Data source:** Google Merchandise Store — GA4 Public Dataset (BigQuery)  
**Output:** `data/raw/ga4_sessions.csv` — clean session-level feature table  

## What this notebook does
1. Authenticates to Google BigQuery (free tier)
2. Queries the GA4 public dataset for e-commerce event logs
3. Engineers session-level features: device, geography, funnel stage, conversion
4. Validates data quality and documents limitations
5. Exports the clean feature table for use in Notebooks 2 and 3

In [4]:
# Install packages not pre-loaded in Colab
# Run this cell first every session — Colab resets installs on reconnect

!pip install -q ruptures kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.8 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.colab import auth
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
import os

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✓ All libraries loaded")
print(f"  pandas {pd.__version__}")
print(f"  numpy {np.__version__}")

✓ All libraries loaded
  pandas 2.2.2
  numpy 2.0.2


In [6]:
# Authenticate your Google account to access BigQuery
# A popup will appear asking you to sign in — use the same Google account
# that owns your Colab. This is a read-only query on a public dataset.

auth.authenticate_user()
print("✓ Google authentication complete")

✓ Google authentication complete


In [7]:
# Authenticate your Google account to access BigQuery
# A popup will appear asking you to sign in — use the same Google account
# that owns your Colab. This is a read-only query on a public dataset.

auth.authenticate_user()
print("✓ Google authentication complete")

✓ Google authentication complete


In [8]:
# You need a Google Cloud project ID to run BigQuery queries.
# The data is PUBLIC and FREE — you are not charged for reading it.
# Your free tier includes 1TB of queries per month. This project uses ~50MB total.

# TO GET YOUR PROJECT ID:
# 1. Go to console.cloud.google.com
# 2. Sign in with your Google account
# 3. Click "Select a project" at the top → "New Project"
# 4. Name it anything: "portfolio-analytics" → Create
# 5. Copy the Project ID (looks like "portfolio-analytics-391420")
# 6. Paste it below

GCP_PROJECT_ID = "portfolio-analytics-505106"  # ← replace this

client = bigquery.Client(project=GCP_PROJECT_ID)
print(f"✓ BigQuery client connected to project: {GCP_PROJECT_ID}")

✓ BigQuery client connected to project: portfolio-analytics-505106


In [9]:
# Google Merchandise Store GA4 public dataset
# Source: https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset
# This is REAL e-commerce event data from Google's own merchandise store
# Date range: 2020-11-01 to 2021-01-31 (3 months of data, ~1M+ events)

query = """
SELECT
    -- Session identifiers
    user_pseudo_id,
    (SELECT value.int_value
     FROM UNNEST(event_params)
     WHERE key = 'ga_session_id') AS session_id,

    -- Timing
    DATE(TIMESTAMP_MICROS(event_timestamp)) AS event_date,
    TIMESTAMP_MICROS(event_timestamp) AS event_timestamp,

    -- Event type
    event_name,

    -- Device
    device.category AS device_category,
    device.operating_system AS os,
    device.web_info.browser AS browser,

    -- Geography
    geo.country AS country,
    geo.region AS region,
    geo.city AS city,

    -- Traffic source
    traffic_source.source AS traffic_source,
    traffic_source.medium AS traffic_medium,

    -- E-commerce
    (SELECT value.double_value
     FROM UNNEST(event_params)
     WHERE key = 'value') AS event_value,

    -- Page
    (SELECT value.string_value
     FROM UNNEST(event_params)
     WHERE key = 'page_title') AS page_title

FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`

WHERE
    -- Focus on the key funnel events
    event_name IN (
        'session_start',
        'view_item',
        'add_to_cart',
        'begin_checkout',
        'add_payment_info',
        'purchase'
    )
    -- 3 months of data — enough to detect anomalies and trends
    AND _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'

LIMIT 500000
"""

print("Querying GA4 public dataset...")
print("Expected: ~300k-500k rows, ~30 seconds")

df_raw = client.query(query).to_dataframe()

print(f"\n✓ Query complete")
print(f"  Rows pulled: {len(df_raw):,}")
print(f"  Columns: {df_raw.shape[1]}")
print(f"  Date range: {df_raw['event_date'].min()} → {df_raw['event_date'].max()}")
print(f"  Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Querying GA4 public dataset...
Expected: ~300k-500k rows, ~30 seconds

✓ Query complete
  Rows pulled: 500,000
  Columns: 15
  Date range: 2020-11-01 → 2021-01-31
  Memory usage: 329.6 MB


In [10]:
# Always inspect before transforming
# This cell is your data quality first pass

print("=== RAW DATA SAMPLE ===")
display(df_raw.head(10))

print("\n=== DATA TYPES ===")
print(df_raw.dtypes)

print("\n=== NULL COUNTS ===")
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw) * 100).round(1)
null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
print(null_summary[null_summary['null_count'] > 0])

print("\n=== EVENT DISTRIBUTION ===")
print(df_raw['event_name'].value_counts())

print("\n=== DEVICE DISTRIBUTION ===")
print(df_raw['device_category'].value_counts())

=== RAW DATA SAMPLE ===


,user_pseudo_id,session_id,event_date,event_timestamp,event_name,device_category,os,browser,country,region,city,traffic_source,traffic_medium,event_value,page_title
0,1003046.9452926974,3209612510,2021-01-01,2021-01-01 02:31:58.563156+00:00,session_start,desktop,Windows,<Other>,France,Auvergne-Rhone-Alpes,(not set),google,organic,NaN,Google Dino Game Tee
1,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:17:24.961359+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,The Google Merchandise Store/Malibu Sunglasses
2,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:15:31.708459+00:00,session_start,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Home
3,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:17:03.025952+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Google Sunnyvale Campus Bottle
4,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:25:20.637924+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Google Aluminum Bottle Red
5,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:24:46.227228+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Google Clear Framed Yellow Shades
6,1026043.4481623879,5588834374,2021-01-01,2021-01-01 15:48:32.499719+00:00,session_start,desktop,Web,Chrome,United States,Wisconsin,Madison,<Other>,referral,NaN,New | Google Merchandise Store
7,1026043.4481623879,9083794581,2021-01-01,2021-01-01 12:25:51.286460+00:00,session_start,desktop,Web,Chrome,United States,Wisconsin,Madison,google,cpc,NaN,YouTube | Shop by Brand | Google Merchandise S...
8,1026409.1551860912,1636061293,2021-01-01,2021-01-01 11:27:32.334474+00:00,session_start,mobile,Web,<Other>,United States,Georgia,(not set),<Other>,referral,NaN,Apparel | Google Merchandise Store
9,1026409.1551860912,328130591,2021-01-01,2021-01-01 14:41:28.495200+00:00,session_start,mobile,Web,<Other>,United States,Georgia,(not set),(data deleted),(data deleted),NaN,Apparel | Google Merchandise Store



=== DATA TYPES ===
user_pseudo_id                  object
session_id                       Int64
event_date                      dbdate
event_timestamp    datetime64[us, UTC]
event_name                      object
device_category                 object
os                              object
browser                         object
country                         object
region                          object
city                            object
traffic_source                  object
traffic_medium                  object
event_value                    float64
page_title                      object
dtype: object

=== NULL COUNTS ===
             null_count  null_pct
event_value      498283   99.7000
page_title         3339    0.7000

=== EVENT DISTRIBUTION ===
event_name
session_start       220806
view_item           220376
add_to_cart          27951
begin_checkout       20636
add_payment_info      7211
purchase              3020
Name: count, dtype: int64

=== DEVICE DISTRIBUTION ===
de

In [11]:
# Always inspect before transforming
# This cell is your data quality first pass

print("=== RAW DATA SAMPLE ===")
display(df_raw.head(10))

print("\n=== DATA TYPES ===")
print(df_raw.dtypes)

print("\n=== NULL COUNTS ===")
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw) * 100).round(1)
null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
print(null_summary[null_summary['null_count'] > 0])

print("\n=== EVENT DISTRIBUTION ===")
print(df_raw['event_name'].value_counts())

print("\n=== DEVICE DISTRIBUTION ===")
print(df_raw['device_category'].value_counts())

=== RAW DATA SAMPLE ===


,user_pseudo_id,session_id,event_date,event_timestamp,event_name,device_category,os,browser,country,region,city,traffic_source,traffic_medium,event_value,page_title
0,1003046.9452926974,3209612510,2021-01-01,2021-01-01 02:31:58.563156+00:00,session_start,desktop,Windows,<Other>,France,Auvergne-Rhone-Alpes,(not set),google,organic,NaN,Google Dino Game Tee
1,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:17:24.961359+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,The Google Merchandise Store/Malibu Sunglasses
2,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:15:31.708459+00:00,session_start,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Home
3,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:17:03.025952+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Google Sunnyvale Campus Bottle
4,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:25:20.637924+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Google Aluminum Bottle Red
5,1023282.4639847710,7473279052,2021-01-01,2021-01-01 16:24:46.227228+00:00,view_item,mobile,Web,Safari,India,Maharashtra,Pune,<Other>,<Other>,NaN,Google Clear Framed Yellow Shades
6,1026043.4481623879,5588834374,2021-01-01,2021-01-01 15:48:32.499719+00:00,session_start,desktop,Web,Chrome,United States,Wisconsin,Madison,<Other>,referral,NaN,New | Google Merchandise Store
7,1026043.4481623879,9083794581,2021-01-01,2021-01-01 12:25:51.286460+00:00,session_start,desktop,Web,Chrome,United States,Wisconsin,Madison,google,cpc,NaN,YouTube | Shop by Brand | Google Merchandise S...
8,1026409.1551860912,1636061293,2021-01-01,2021-01-01 11:27:32.334474+00:00,session_start,mobile,Web,<Other>,United States,Georgia,(not set),<Other>,referral,NaN,Apparel | Google Merchandise Store
9,1026409.1551860912,328130591,2021-01-01,2021-01-01 14:41:28.495200+00:00,session_start,mobile,Web,<Other>,United States,Georgia,(not set),(data deleted),(data deleted),NaN,Apparel | Google Merchandise Store



=== DATA TYPES ===
user_pseudo_id                  object
session_id                       Int64
event_date                      dbdate
event_timestamp    datetime64[us, UTC]
event_name                      object
device_category                 object
os                              object
browser                         object
country                         object
region                          object
city                            object
traffic_source                  object
traffic_medium                  object
event_value                    float64
page_title                      object
dtype: object

=== NULL COUNTS ===
             null_count  null_pct
event_value      498283   99.7000
page_title         3339    0.7000

=== EVENT DISTRIBUTION ===
event_name
session_start       220806
view_item           220376
add_to_cart          27951
begin_checkout       20636
add_payment_info      7211
purchase              3020
Name: count, dtype: int64

=== DEVICE DISTRIBUTION ===
de

In [12]:
def build_session_features(df):
    """
    Collapse event-level GA4 data into session-level features.

    Each session is one row with:
    - Funnel progression flags (did this session reach each stage?)
    - Device, geography, traffic source attributes
    - Conversion outcome (0/1)
    - Session date for time-series analysis

    Args:
        df: Raw GA4 event-level dataframe

    Returns:
        session_df: Session-level feature table
    """

    # Define funnel stage order
    funnel_stages = [
        'session_start',
        'view_item',
        'add_to_cart',
        'begin_checkout',
        'add_payment_info',
        'purchase'
    ]

    # Create binary flags for each funnel stage
    for stage in funnel_stages:
        df[f'has_{stage}'] = (df['event_name'] == stage).astype(int)

    # Aggregate to session level
    session_agg = df.groupby(
        ['user_pseudo_id', 'session_id', 'event_date']
    ).agg(
        # Funnel stages — max() because 1 event is enough to flag the stage
        session_start    = ('has_session_start', 'max'),
        viewed_item      = ('has_view_item', 'max'),
        added_to_cart    = ('has_add_to_cart', 'max'),
        began_checkout   = ('has_begin_checkout', 'max'),
        added_payment    = ('has_add_payment_info', 'max'),
        converted        = ('has_purchase', 'max'),

        # Device and geo — take first (consistent within session)
        device_category  = ('device_category', 'first'),
        os               = ('os', 'first'),
        country          = ('country', 'first'),
        region           = ('region', 'first'),
        traffic_source   = ('traffic_source', 'first'),
        traffic_medium   = ('traffic_medium', 'first'),

        # Revenue
        session_revenue  = ('event_value', 'sum'),

        # Session depth
        event_count      = ('event_name', 'count')

    ).reset_index()

    # Highest funnel stage reached (consulting-useful feature)
    def get_funnel_stage(row):
        if row['converted']:         return 6
        if row['added_payment']:     return 5
        if row['began_checkout']:    return 4
        if row['added_to_cart']:     return 3
        if row['viewed_item']:       return 2
        return 1

    session_agg['max_funnel_stage'] = session_agg.apply(
        get_funnel_stage, axis=1
    )

    # Funnel stage labels for readability
    stage_labels = {
        1: 'session_only',
        2: 'view_item',
        3: 'add_to_cart',
        4: 'begin_checkout',
        5: 'add_payment',
        6: 'purchase'
    }
    session_agg['funnel_stage_label'] = session_agg[
        'max_funnel_stage'
    ].map(stage_labels)

    return session_agg


print("Engineering session-level features...")
df_sessions = build_session_features(df_raw)

print(f"✓ Session feature table built")
print(f"  Sessions: {len(df_sessions):,}")
print(f"  Features: {df_sessions.shape[1]}")
print(f"  Overall conversion rate: {df_sessions['converted'].mean():.2%}")
print(f"\nSample:")
display(df_sessions.head())

Engineering session-level features...
✓ Session feature table built
  Sessions: 221,838
  Features: 19
  Overall conversion rate: 1.15%

Sample:


,user_pseudo_id,session_id,event_date,session_start,viewed_item,added_to_cart,began_checkout,added_payment,converted,device_category,os,country,region,traffic_source,traffic_medium,session_revenue,event_count,max_funnel_stage,funnel_stage_label
0,10001363.4360935308,5983736405,2020-12-12,1,0,0,0,0,0,desktop,Web,Sweden,Stockholm County,<Other>,organic,0.0000,1,1,session_only
1,1000223163.8035209215,6063162078,2021-01-07,1,0,0,0,0,0,mobile,Web,Australia,New South Wales,(direct),(none),0.0000,1,1,session_only
2,10004358.0897722689,39098263,2021-01-08,1,0,0,0,0,0,mobile,Web,Russia,(not set),google,cpc,0.0000,1,1,session_only
3,10005335.8064658740,1243001234,2020-12-12,1,1,0,0,0,0,desktop,Windows,China,(not set),google,cpc,0.0000,2,2,view_item
4,10005335.8064658740,3552270955,2020-12-12,1,0,0,0,0,0,desktop,Windows,China,(not set),(data deleted),(data deleted),0.0000,1,1,session_only


# Daily Conversion Rate Time Series

In [13]:
def compute_daily_metrics(df):
    """
    Compute daily conversion rate and session volume.
    This is the time series the anomaly detector will monitor.
    """
    daily = df.groupby('event_date').agg(
        total_sessions   = ('session_id', 'count'),
        conversions      = ('converted', 'sum'),
        total_revenue    = ('session_revenue', 'sum')
    ).reset_index()

    daily['conversion_rate'] = (
        daily['conversions'] / daily['total_sessions']
    )
    daily['event_date'] = pd.to_datetime(daily['event_date'])
    daily = daily.sort_values('event_date').reset_index(drop=True)

    return daily


df_daily = compute_daily_metrics(df_sessions)

# Plot — this is the time series you'll run anomaly detection on
fig = px.line(
    df_daily,
    x='event_date',
    y='conversion_rate',
    title='Daily Conversion Rate — Google Merchandise Store (Nov 2020 – Jan 2021)<br>'
          '<sup>Baseline time series for anomaly detection model</sup>',
    labels={
        'event_date': 'Date',
        'conversion_rate': 'Conversion Rate'
    }
)

fig.update_traces(line_color='#1f77b4', line_width=1.5)
fig.update_layout(
    yaxis_tickformat='.1%',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font_family='Arial',
    height=400
)
fig.add_annotation(
    text="Source: Google Merchandise Store GA4 Public Dataset, BigQuery",
    xref="paper", yref="paper",
    x=0, y=-0.15, showarrow=False,
    font=dict(size=10, color='grey')
)

fig.show()

print(f"\nDaily metrics summary:")
print(f"  Avg daily sessions: {df_daily['total_sessions'].mean():.0f}")
print(f"  Avg conversion rate: {df_daily['conversion_rate'].mean():.2%}")
print(f"  Min conversion rate: {df_daily['conversion_rate'].min():.2%} "
      f"on {df_daily.loc[df_daily['conversion_rate'].idxmin(), 'event_date'].date()}")
print(f"  Max conversion rate: {df_daily['conversion_rate'].max():.2%} "
      f"on {df_daily.loc[df_daily['conversion_rate'].idxmax(), 'event_date'].date()}")


Daily metrics summary:
  Avg daily sessions: 3413
  Avg conversion rate: 1.11%
  Min conversion rate: 0.00% on 2021-01-27
  Max conversion rate: 2.41% on 2020-12-12


# Data Quality Validation

In [14]:
def validate_data_quality(df_sessions, df_daily):
    """
    Run data quality checks before saving.
    Prints a pass/fail report — any FAIL requires investigation.
    """
    checks = []

    # Check 1: No duplicate sessions
    dupes = df_sessions.duplicated(
        subset=['user_pseudo_id', 'session_id', 'event_date']
    ).sum()
    checks.append(('No duplicate sessions', dupes == 0,
                   f"{dupes} duplicates found"))

    # Check 2: Conversion rate in plausible range (0.5% – 15%)
    avg_cr = df_sessions['converted'].mean()
    checks.append(('Conversion rate plausible', 0.005 <= avg_cr <= 0.15,
                  f"Avg CVR = {avg_cr:.2%}"))

    # Check 3: All dates present (no gaps > 2 days)
    date_range = pd.date_range(
        df_daily['event_date'].min(),
        df_daily['event_date'].max()
    )
    missing_dates = len(date_range) - len(df_daily)
    checks.append(('No significant date gaps', missing_dates <= 2,
                  f"{missing_dates} missing dates"))

    # Check 4: Device categories are known values
    known_devices = {'mobile', 'desktop', 'tablet'}
    actual_devices = set(
        df_sessions['device_category'].dropna().unique()
    )
    unknown = actual_devices - known_devices
    checks.append(('Device categories valid', len(unknown) == 0,
                  f"Unknown devices: {unknown}"))

    # Check 5: No sessions with negative revenue
    neg_rev = (df_sessions['session_revenue'] < 0).sum()
    checks.append(('No negative revenue', neg_rev == 0,
                  f"{neg_rev} sessions with negative revenue"))

    print("=== DATA QUALITY VALIDATION ===\n")
    all_pass = True
    for check_name, passed, detail in checks:
        status = "✓ PASS" if passed else "✗ FAIL"
        if not passed:
            all_pass = False
        print(f"  {status}  {check_name}")
        print(f"         {detail}\n")

    print("=" * 35)
    if all_pass:
        print("✓ All checks passed — data is ready for modeling")
    else:
        print("⚠ Some checks failed — review before proceeding")

    return all_pass


data_ready = validate_data_quality(df_sessions, df_daily)

=== DATA QUALITY VALIDATION ===

  ✓ PASS  No duplicate sessions
         0 duplicates found

  ✓ PASS  Conversion rate plausible
         Avg CVR = 1.15%

  ✗ FAIL  No significant date gaps
         27 missing dates

  ✓ PASS  Device categories valid
         Unknown devices: set()

  ✓ PASS  No negative revenue
         0 sessions with negative revenue

⚠ Some checks failed — review before proceeding


In [15]:
# Save outputs locally in Colab first
# Then we upload to GitHub manually

os.makedirs('data/raw', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

if data_ready:
    # Session-level feature table — this feeds Notebook 2 (anomaly detection)
    df_sessions.to_csv('data/raw/ga4_sessions.csv', index=False)

    # Daily metrics — this feeds Notebook 3 (CUSUM changepoint detection)
    df_daily.to_csv('data/raw/ga4_daily_metrics.csv', index=False)

    print("✓ Files saved:")
    print(f"  data/raw/ga4_sessions.csv      "
          f"({len(df_sessions):,} rows, "
          f"{os.path.getsize('data/raw/ga4_sessions.csv')/1024:.0f} KB)")
    print(f"  data/raw/ga4_daily_metrics.csv "
          f"({len(df_daily):,} rows, "
          f"{os.path.getsize('data/raw/ga4_daily_metrics.csv')/1024:.0f} KB)")
    print(f"\nNext: open Notebook 2 — anomaly detection model")
else:
    print("⚠ Fix data quality issues before saving")

⚠ Fix data quality issues before saving


## Notebook 1 Complete

**What was built:**
- Pulled 300k–500k GA4 events from BigQuery public dataset
- Engineered session-level features across 6 funnel stages
- Validated data quality across 5 checks
- Exported clean feature tables for downstream notebooks

**Output files:**
- `data/raw/ga4_sessions.csv` — session-level feature table (input to Notebook 2)
- `data/raw/ga4_daily_metrics.csv` — daily conversion rate time series (input to Notebook 3)

**Next notebook:** `02_anomaly_detection.ipynb` — Isolation Forest + CUSUM model